In [29]:
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_groq import ChatGroq
from dotenv import load_dotenv

load_dotenv()

True

In [14]:
chunks = [
    "Microsoft acquired GitHub for 7.5 billion dollars in 2018.",
    "Tesla Cybertruck production ramp begins in 2024.",
    "Google is a large technology company with global operations.",
    "Tesla reported strong quarterly results. Tesla continues to lead in electric vehicles. Tesla announced new manufacturing facilities.",
    "SpaceX develops Starship rockets for Mars missions.",
    "The tech giant acquired the code repository platform for software development.",
    "NVIDIA designs Starship architecture for their new GPUs.",
    "Tesla Tesla Tesla financial quarterly results improved significantly.",
    "Cybertruck reservations exceeded company expectations.",
    "Microsoft is a large technology company with global operations.", 
    "Apple announced new iPhone features for developers.",
    "The apple orchard harvest was excellent this year.",
    "Python programming language is widely used in AI.",
    "The python snake can grow up to 20 feet long.",
    "Java coffee beans are imported from Indonesia.", 
    "Java programming requires understanding of object-oriented concepts.",
    "Orange juice sales increased during winter months.",
    "Orange County reported new housing developments."
]

In [ ]:
documents = []
for i,chunk in enumerate(chunks):
    doc = Document(
        page_content=chunk,
        meta_data = {"source":f"chunk_{i}"}
    )
    documents.append(doc)

In [21]:
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = Chroma.from_documents(
    documents = documents,
    embedding = embedding_model,
    collection_metadata={"hnsw:space":"cosine"}
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4018.02it/s]


In [ ]:
print("Vector Search")
print()
vector_retriever = vector_store.as_retriever(search_kwargs={"k":3})
test_query = "space exploration company"

test_docs = vector_retriever.invoke(test_query)
for i,docs in enumerate(test_docs):
    print(f"Document{i}: {docs.page_content}")

Document0: SpaceX develops Starship rockets for Mars missions.
Document1: SpaceX develops Starship rockets for Mars missions.
Document2: Google is a large technology company with global operations.


In [24]:
print("BM25 Retriever")

bm25_retriever = BM25Retriever.from_documents(documents=documents)
bm25_retriever.k = 3

BM25 Retriever


In [ ]:
test_query = "Tesla"
test_docs = bm25_retriever.invoke(test_query)
for i,doc in enumerate(test_docs,1):
    print(f"Document{i}: {doc}")

Document0: page_content='Tesla Tesla Tesla financial quarterly results improved significantly.'
Document1: page_content='Tesla reported strong quarterly results. Tesla continues to lead in electric vehicles. Tesla announced new manufacturing facilities.'
Document2: page_content='Tesla Cybertruck production ramp begins in 2024.'


In [27]:
print("EnsembleRetriever")

hybrid_retriever = EnsembleRetriever(
    retrievers=[vector_retriever,bm25_retriever],
    weights=[0.7,0.3]
)

test_query = "purchase cost 7.5 billion"
hybrid_docs = hybrid_retriever.invoke(test_query)

for i,doc in enumerate(hybrid_docs,1):
    print(f"Document{i}: {doc.page_content}")

EnsembleRetriever
Document1: Microsoft acquired GitHub for 7.5 billion dollars in 2018.
Document2: Microsoft is a large technology company with global operations.
Document3: Orange County reported new housing developments.
Document4: Orange juice sales increased during winter months.


In [32]:
from langchain_core.messages import HumanMessage, SystemMessage
import os
combined_input = f"""Based on the following documents, please answer this question: {test_query}

Documents:
{chr(10).join([f"- {doc.page_content}" for doc in hybrid_docs])}

Please provide a clear, helpful answer using only the information from these documents. If you can't find the answer in the documents, say "I don't have enough information to answer that question based on the provided documents."
"""

# Create a ChatOpenAI model
model = ChatGroq(model="openai/gpt-oss-120b",api_key=os.getenv("GROQ_API_KEY"))

# Define the messages for the model
messages = [
    SystemMessage(content="You are a helpful assistant."),
    HumanMessage(content=combined_input),
]

# Invoke the model with the combined input
result = model.invoke(messages)

# Display the full result and content only
print("\n--- Generated Response ---")
print("Full result:")
# print(result)
# print("Content only:")
print(result.content)


--- Generated Response ---
Full result:
The purchase cost of $7.5 billion refers to Microsoft’s acquisition of GitHub in 2018.
